# Akili V4 - External Structural Generalisation Colab

**Protocol:** `akili-v4-external-structural-generalisation-v1`

This notebook is phase locked:

- `development` → seeds 1, 2, 3
- `heldout` → seeds 4, 5, 6

Run development first. Freeze the implementation and criteria before setting the held-out phase. The notebook embeds the extracted V3.1.1a `AkiliCore`; it does not edit the frozen V3.1.1a evidence files.

## Scientific question
Can the same Akili lifecycle and template mechanism generalise to six new structural families: validation, serialisation, structured logging, configuration precedence, batching and artifact naming?

## Arms
Stateless, bounded ICL, immediate-template-only, and full Akili.


> **FROZEN HELD-OUT COPY:** seeds 4, 5, 6. Do not edit cells, prompts, templates, budgets, scorer, or criteria.


In [ ]:
# FROZEN HELD-OUT CONFIG — seeds 4, 5, 6 only. Do not edit.
import os, sys, json, subprocess, hashlib
from pathlib import Path

PHASE = "heldout"
LOCKED = {"development": [1, 2, 3], "heldout": [4, 5, 6]}
SEEDS = LOCKED[PHASE]
assert PHASE == "heldout"
assert SEEDS == [4, 5, 6]
MODEL = os.environ.get("AKILI_MODEL", "Qwen/Qwen3-4B")
DRIVE_ROOT = os.environ.get("AKILI_DRIVE_ROOT", "/content/drive/MyDrive/AKILI_CL")
SESSION = "akili_v4_external_generalisation_heldout_seeds_4_5_6"
WORK_ROOT = Path(os.environ.get("AKILI_WORK_ROOT", "/content/akili_v4_external_generalisation"))
OUT = Path(DRIVE_ROOT) / SESSION
print("FROZEN V4 HELD-OUT: seeds 4, 5, 6")
print({"protocol": "akili-v4-external-structural-generalisation-v1", "phase": PHASE, "seeds": SEEDS, "model": MODEL, "session": SESSION})


In [ ]:
# Install only the pinned packages required for 4-bit Qwen inference.
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "transformers==4.53.2", "accelerate==1.8.1", "bitsandbytes==0.46.1"
], check=True, timeout=480)


In [ ]:
# Mount Drive and validate CUDA before downloading the model.
from google.colab import drive
drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "CUDA GPU required — stop before model download"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print("WORK_ROOT:", WORK_ROOT)
print("OUT:", OUT)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Embedded V3.1.1a core source — written verbatim for import.
CORE_SOURCE = '# =============================================================================\n# V3.1.1 SYSTEMS — independent instances, bounded ICL state, scoped Akili state.\n# =============================================================================\nimport copy\nimport hashlib\nimport json\nimport re\nimport uuid\n\n\ndef truncate_to_budget(text, budget_tokens, count_fn):\n    text = text or ""\n    if budget_tokens <= 0:\n        return "", 0\n    if count_fn(text) <= budget_tokens:\n        return text, count_fn(text)\n    lo, hi = 0, len(text)\n    while lo < hi:\n        mid = (lo + hi + 1) // 2\n        if count_fn(text[:mid]) <= budget_tokens:\n            lo = mid\n        else:\n            hi = mid - 1\n    fitted = text[:lo]\n    used = count_fn(fitted)\n    while fitted and used > budget_tokens:\n        fitted = fitted[:-1]\n        used = count_fn(fitted)\n    return fitted, used\n\n\ndef fit_newest_to_budget(parts_newest_first, budget_tokens, count_fn, header=""):\n    chosen = []\n    base = header\n    for part in parts_newest_first:\n        candidate_parts = chosen + [part]\n        body = "\\n\\n".join(candidate_parts)\n        candidate = (base + "\\n" + body).strip() if base else body\n        if count_fn(candidate) <= budget_tokens:\n            chosen.append(part)\n            continue\n        remaining = budget_tokens - count_fn((base + "\\n" + "\\n\\n".join(chosen)).strip())\n        if remaining > 0:\n            fitted, _ = truncate_to_budget(part, remaining, count_fn)\n            if fitted:\n                chosen.append(fitted)\n        break\n    text = ((base + "\\n") if base else "") + "\\n\\n".join(chosen)\n    text = text.strip()\n    text, used = truncate_to_budget(text, budget_tokens, count_fn)\n    return text, used\n\n\nclass Stateless:\n    name = "stateless"\n\n    def __init__(self, count_fn):\n        self.count = count_fn\n        self.instance_id = uuid.uuid4().hex\n\n    def memory_prompt(self):\n        return ""\n\n    def observe_episode(self, summary, features, entity_hint, outcome):\n        return None\n\n    def report(self):\n        return {\n            "instance_id": self.instance_id,\n            "stored_memory_chars": 0,\n            "stored_memory_tokens": 0,\n            "active_rules": 0,\n            "admitted_total": 0,\n            "superseded": 0,\n        }\n\n\nclass BoundedICL:\n    name = "bounded_icl"\n\n    def __init__(self, budget_tokens, count_fn):\n        self.budget = budget_tokens\n        self.count = count_fn\n        self.history = []\n        self.instance_id = uuid.uuid4().hex\n\n    def _render(self):\n        if not self.history:\n            return "", 0\n        return fit_newest_to_budget(\n            list(reversed(self.history)), self.budget, self.count,\n            header="RECENT RAW EPISODE HISTORY:",\n        )\n\n    def _trim_state(self):\n        while len(self.history) > 1:\n            blob, used = self._render()\n            if used <= self.budget and self.count("\\n\\n".join(self.history)) <= self.budget:\n                break\n            self.history.pop(0)\n        if self.history:\n            latest, _ = truncate_to_budget(self.history[-1], self.budget, self.count)\n            self.history[-1] = latest\n\n    def memory_prompt(self):\n        return self._render()[0]\n\n    def observe_episode(self, summary, features, entity_hint, outcome):\n        self.history.append(str(summary))\n        self._trim_state()\n\n    def report(self):\n        blob = "\\n\\n".join(self.history)\n        prompt, prompt_tokens = self._render()\n        return {\n            "instance_id": self.instance_id,\n            "history_entries": len(self.history),\n            "stored_memory_chars": len(blob),\n            "stored_memory_tokens": self.count(blob),\n            "retrievable_tokens": prompt_tokens,\n            "state_bounded": self.count(blob) <= self.budget,\n            "active_rules": 0,\n            "admitted_total": 0,\n            "superseded": 0,\n        }\n\n\n_NAME_OK = re.compile(r"^[A-Za-z][A-Za-z0-9 .\'\\-_]{0,31}$")\n_NAME_BAD_WORDS = frozenset({"policy", "stage", "schedule", "instance"})\n\n\n\nclass AkiliCore:\n    name = "akili"\n\n    def __init__(self, budget_tokens, count_fn, derive_fn, min_evidence=2,\n                 max_profiles=16, max_versions=48, recent_window=10,\n                 max_stats_keys=96, max_retrieval_log=512):\n        self.budget = budget_tokens\n        self.count = count_fn\n        self.derive_fn = derive_fn\n        self.min_evidence = min_evidence\n        self.max_profiles = max_profiles\n        self.max_versions = max_versions\n        self.recent_window = recent_window\n        self.max_stats_keys = max_stats_keys\n        self.max_retrieval_log = max_retrieval_log\n        self.instance_id = uuid.uuid4().hex\n        self.profiles = {}\n        self.current_key = None\n        self.audit = []\n        self.episode_count = 0\n        self.cross_scope_retrievals = 0\n        self.retrieval_log = []\n        self._audit("RUN_START", {})\n\n    def _audit(self, event, payload):\n        body = {\n            "seq": len(self.audit), "event": event, "episode": self.episode_count,\n            "profile": self.current_key, "payload": payload,\n            "prev_hash": self.audit[-1]["hash"] if self.audit else "GENESIS",\n        }\n        body["hash"] = hashlib.sha256(\n            json.dumps(body, sort_keys=True, default=str).encode("utf-8")\n        ).hexdigest()\n        self.audit.append(body)\n\n    def validate_chain(self):\n        prev = "GENESIS"\n        for entry in self.audit:\n            body = {k: v for k, v in entry.items() if k != "hash"}\n            if body["prev_hash"] != prev:\n                return False\n            expected = hashlib.sha256(\n                json.dumps(body, sort_keys=True, default=str).encode("utf-8")\n            ).hexdigest()\n            if expected != entry["hash"]:\n                return False\n            prev = entry["hash"]\n        return True\n\n    def _new_profile(self, key, display):\n        assert len(self.profiles) < self.max_profiles, "Akili profile bound exceeded"\n        self.profiles[key] = {\n            "key": key, "display": display, "status": "ACTIVE", "episodes": 0,\n            "stats": {}, "recent_features": [], "versions": [], "version_seq": 0,\n            "active_by_family": {}, "times_activated": 1,\n        }\n\n    def _accept_name(self, raw):\n        name = str(raw or "").strip()\n        if not _NAME_OK.match(name):\n            return None\n        if any(word in name.lower() for word in _NAME_BAD_WORDS):\n            return None\n        return name\n\n    def switch_entity(self, raw_name):\n        name = self._accept_name(raw_name)\n        if name is None:\n            key = "anon:" + hashlib.sha256(str(raw_name).encode()).hexdigest()[:16]\n            display = None\n        else:\n            key = "name:" + hashlib.sha256(name.lower().encode()).hexdigest()[:16]\n            display = name\n        if key == self.current_key:\n            return\n        previous = self.current_key\n        if previous is not None:\n            self.profiles[previous]["status"] = "DORMANT"\n        if key in self.profiles:\n            self.profiles[key]["status"] = "ACTIVE"\n            self.profiles[key]["times_activated"] += 1\n            self.current_key = key\n            self._audit("PROFILE_REACTIVATED", {"from": previous, "to": key, "display": display})\n        else:\n            self._new_profile(key, display)\n            self.current_key = key\n            self._audit("PROFILE_CREATED", {"from": previous, "to": key, "display": display})\n\n    def _append_version(self, profile, version):\n        if len(profile["versions"]) >= self.max_versions:\n            removable = next((i for i, item in enumerate(profile["versions"])\n                              if item["status"] in {"REJECTED", "SUPERSEDED"}), None)\n            assert removable is not None, "Akili version bound exceeded with no removable record"\n            removed = profile["versions"].pop(removable)\n            self._audit("VERSION_EVICTED", {"version_hash": removed["hash"], "status": removed["status"]})\n        profile["versions"].append(version)\n\n    def observe_episode(self, summary, features, entity_hint, outcome):\n        if self.current_key is None:\n            self.switch_entity(entity_hint or "unnamed")\n        self.episode_count += 1\n        profile = self.profiles[self.current_key]\n        profile["episodes"] += 1\n        features = dict(features or {})\n        profile["recent_features"].append(features)\n        profile["recent_features"] = profile["recent_features"][-self.recent_window:]\n        for key, value in features.items():\n            if key not in profile["stats"]:\n                assert len(profile["stats"]) < self.max_stats_keys, "Akili stats-key bound exceeded"\n            if isinstance(value, (int, float)):\n                profile["stats"][key] = profile["stats"].get(key, 0) + value\n            elif key.startswith("latest_"):\n                profile["stats"][key] = value\n            else:\n                profile["stats"].setdefault(key, value)\n        self._audit("EPISODE_RECORDED", {"features": features, "outcome": outcome})\n        candidates = self.derive_fn(profile["stats"], profile, features)\n        if not candidates:\n            return\n        if isinstance(candidates, dict):\n            candidates = [candidates]\n        for candidate in candidates:\n            self._process_candidate(profile, candidate)\n\n    @staticmethod\n    def _version_hash(version):\n        body = {k: v for k, v in version.items() if k != "hash"}\n        return hashlib.sha256(json.dumps(body, sort_keys=True, default=str).encode("utf-8")).hexdigest()\n\n    def _process_candidate(self, profile, candidate):\n        family = candidate.get("family", "general")\n        lifecycle = candidate.get("lifecycle_state", "VERIFIED")\n        assert lifecycle in {"PROVISIONAL", "VERIFIED", "CONFLICTED"}\n        active = profile["active_by_family"].get(family)\n        same_rule = active is not None and active["rule"] == candidate["rule"]\n        same_state = active is not None and active.get("lifecycle_state", active["status"]) == lifecycle\n\n        schema_template = candidate.get("schema_template")\n        template_metadata = copy.deepcopy(candidate.get("template_metadata") or {})\n        template_hash = hashlib.sha256(schema_template.encode("utf-8")).hexdigest() if schema_template else None\n        verified_exemplar = candidate.get("verified_exemplar") or candidate.get("exemplar")\n        verified_structural_exemplar = candidate.get("verified_structural_exemplar")\n        exemplar_shape_key = candidate.get("exemplar_shape_key")\n\n        if same_rule and same_state:\n            changed = False\n            old_hash = active["hash"]\n            if schema_template and schema_template != active.get("schema_template"):\n                active["schema_template"] = schema_template\n                active["schema_template_hash"] = template_hash\n                active["template_metadata"] = template_metadata\n                changed = True\n                self._audit("SCHEMA_TEMPLATE_ADDED", {\n                    "family": family, "template_hash": template_hash,\n                    "shape_key": template_metadata.get("shape_key"),\n                    "source": template_metadata.get("source"),\n                })\n            if verified_exemplar and exemplar_shape_key:\n                exemplars = active.setdefault("verified_exemplars", {})\n                previous = exemplars.get(exemplar_shape_key)\n                exemplar_hash = hashlib.sha256(verified_exemplar.encode("utf-8")).hexdigest()\n                if not previous or previous.get("hash") != exemplar_hash:\n                    structural_hash = (\n                        hashlib.sha256(verified_structural_exemplar.encode("utf-8")).hexdigest()\n                        if verified_structural_exemplar else None\n                    )\n                    exemplars[exemplar_shape_key] = {\n                        "content": verified_exemplar,\n                        "hash": exemplar_hash,\n                        "structural_content": verified_structural_exemplar,\n                        "structural_hash": structural_hash,\n                        "shape_key": exemplar_shape_key,\n                        "verified": True,\n                    }\n                    changed = True\n                    self._audit("VERIFIED_EXEMPLAR_ADDED", {\n                        "family": family, "shape_key": exemplar_shape_key,\n                        "exemplar_hash": exemplar_hash,\n                    })\n            if changed:\n                active["hash"] = self._version_hash(active)\n                self._audit("ACTIVE_RECORD_UPDATED", {\n                    "family": family, "old_hash": old_hash, "new_hash": active["hash"],\n                })\n            else:\n                self._audit("CANDIDATE_DUPLICATE", {\n                    "family": family, "lifecycle_state": lifecycle,\n                    "evidence": candidate.get("evidence", {}),\n                })\n            return\n\n        min_gap = int(candidate.get("min_transition_gap", 0))\n        if active is not None and profile["episodes"] - active["episode"] < min_gap:\n            self._audit("TRANSITION_DEFERRED", {\n                "family": family, "lifecycle_state": lifecycle,\n                "episodes_since_active": profile["episodes"] - active["episode"],\n                "evidence": candidate.get("evidence", {}),\n            })\n            return\n\n        audit_candidate = {\n            k: v for k, v in candidate.items()\n            if k not in {"schema_template", "verified_exemplar", "verified_structural_exemplar", "exemplar"}\n        }\n        audit_candidate["schema_template_hash"] = template_hash\n        if verified_exemplar:\n            audit_candidate["verified_exemplar_hash"] = hashlib.sha256(verified_exemplar.encode("utf-8")).hexdigest()\n        if verified_structural_exemplar:\n            audit_candidate["verified_structural_exemplar_hash"] = hashlib.sha256(\n                verified_structural_exemplar.encode("utf-8")\n            ).hexdigest()\n        self._audit("CANDIDATE_CREATED", audit_candidate)\n        admitted, reason = self._gate(candidate)\n        profile["version_seq"] += 1\n        verified_exemplars = {}\n        if verified_exemplar and exemplar_shape_key:\n            exemplar_hash = hashlib.sha256(verified_exemplar.encode("utf-8")).hexdigest()\n            structural_hash = (\n                hashlib.sha256(verified_structural_exemplar.encode("utf-8")).hexdigest()\n                if verified_structural_exemplar else None\n            )\n            verified_exemplars[exemplar_shape_key] = {\n                "content": verified_exemplar,\n                "hash": exemplar_hash,\n                "structural_content": verified_structural_exemplar,\n                "structural_hash": structural_hash,\n                "shape_key": exemplar_shape_key,\n                "verified": True,\n            }\n        version = {\n            "version": profile["version_seq"], "family": family, "rule": candidate["rule"],\n            "evidence": candidate.get("evidence", {}),\n            "status": lifecycle if admitted else "REJECTED",\n            "lifecycle_state": lifecycle if admitted else "REJECTED",\n            "gate_reason": reason, "episode": profile["episodes"],\n            "decision_label": candidate.get("decision_label"),\n            "schema_template": schema_template,\n            "schema_template_hash": template_hash,\n            "template_metadata": template_metadata,\n            "verified_exemplars": verified_exemplars,\n        }\n        version["hash"] = self._version_hash(version)\n        self._append_version(profile, version)\n        if admitted:\n            if active is not None:\n                old_state = active.get("lifecycle_state", active.get("status"))\n                active["status"] = "SUPERSEDED"\n                self._audit("SUPERSEDED", {\n                    "family": family, "old_rule": active["rule"], "new_rule": candidate["rule"],\n                    "old_state": old_state, "new_state": lifecycle,\n                    "evidence": candidate.get("evidence", {}),\n                })\n            profile["active_by_family"][family] = version\n            self._audit("ADMITTED", {\n                "family": family, "rule": candidate["rule"], "lifecycle_state": lifecycle,\n                "decision_label": candidate.get("decision_label"),\n                "evidence": candidate.get("evidence", {}), "reason": reason,\n                "schema_template_hash": template_hash,\n                "template_source": template_metadata.get("source"),\n                "template_shape_key": template_metadata.get("shape_key"),\n                "verified_exemplar_hashes": {\n                    key: value["hash"] for key, value in verified_exemplars.items()\n                },\n            })\n        else:\n            self._audit("REJECTED", {"family": family, "evidence": candidate.get("evidence", {}), "reason": reason})\n\n    def _gate(self, candidate):\n        evidence = candidate.get("evidence", {})\n        count = int(evidence.get("count", 0))\n        threshold = int(candidate.get("admit_threshold", self.min_evidence))\n        if count < threshold:\n            return False, f"evidence {count} < {threshold}"\n        return True, f"evidence {count} >= {threshold}"\n\n    def active_record(self, family):\n        if self.current_key is None:\n            return None\n        return self.profiles[self.current_key]["active_by_family"].get(family)\n\n    def retrieval_context(self, family=None, operation_shape=None):\n        details = {\n            "requested_family": family,\n            "operation_shape": operation_shape,\n            "rule_available": False,\n            "persistent_rule_available": False,\n            "rule_retrieved": False,\n            "schema_template_available": False,\n            "schema_template_retrieved": False,\n            "persistent_schema_template_retrieved": False,\n            "current_instruction_template_available": False,\n            "current_instruction_template_retrieved": False,\n            "verified_exemplar_available": False,\n            "verified_exemplar_retrieved": False,\n            "verified_structural_exemplar_available": False,\n            "verified_structural_exemplar_retrieved": False,\n            "raw_verified_exemplar_retrieved": False,\n            "verified_exemplar_suppressed_by_template": False,\n            "shape_match": False,\n            "memory_provenance": "NONE",\n            "primary_memory_class": "NO_MEMORY",\n            "applicable_rule_state": None,\n            "applicable_decision_label": None,\n            "applicable_rule_hash": None,\n            "template_hash_at_decision": None,\n            "template_source_at_decision": None,\n            "prior_active_convention_suppressed": False,\n        }\n        if self.current_key is None:\n            return "", details\n        profile = self.profiles[self.current_key]\n        if family is None:\n            actives = list(profile["active_by_family"].values())\n        else:\n            active = profile["active_by_family"].get(family)\n            actives = [active] if active is not None else []\n        if not actives:\n            self.retrieval_log.append({\n                "episode": self.episode_count, "requested": self.current_key,\n                "retrieved": self.current_key, "tokens": 0, **details,\n            })\n            self.retrieval_log = self.retrieval_log[-self.max_retrieval_log:]\n            self._audit("PROFILE_RETRIEVED", {"retrieved": self.current_key, "tokens": 0, **details})\n            return "", details\n\n        lines = [f"=== AKILI PROFILE: {profile[\'display\'] or \'unnamed\'} | episodes={profile[\'episodes\']} ==="]\n        provenance_rank = {\n            "NONE": 0, "RULE": 1, "RAW_VERIFIED_EXEMPLAR": 2,\n            "VERIFIED_STRUCTURAL_EXEMPLAR": 3, "SCHEMA_TEMPLATE": 4,\n        }\n        for item in actives:\n            state = item.get("lifecycle_state", item["status"])\n            lines.append(f"- [{item[\'family\']} | {state}] {item[\'rule\']}")\n            details["rule_available"] = True\n            details["persistent_rule_available"] = True\n            details["rule_retrieved"] = True\n            details["applicable_rule_state"] = state if family else details["applicable_rule_state"]\n            details["applicable_decision_label"] = item.get("decision_label") if family else details["applicable_decision_label"]\n            details["applicable_rule_hash"] = item.get("hash") if family else details["applicable_rule_hash"]\n            if provenance_rank["RULE"] > provenance_rank.get(details["memory_provenance"], 0):\n                details["memory_provenance"] = "RULE"\n                details["primary_memory_class"] = "RULE_ONLY"\n            ev = item.get("evidence") or {}\n            compact = {k: ev[k] for k in (\n                "malicious_count", "benign_count", "recent_labels", "count", "signature"\n            ) if k in ev}\n            if compact:\n                lines.append("  Evidence: " + json.dumps(compact, sort_keys=True))\n\n            exemplars = item.get("verified_exemplars") or {}\n            exact_exemplar = exemplars.get(operation_shape) if operation_shape else None\n            structural = (exact_exemplar or {}).get("structural_content") if exact_exemplar else None\n            details["verified_exemplar_available"] = bool(exact_exemplar)\n            details["verified_structural_exemplar_available"] = bool(structural)\n            template = item.get("schema_template")\n            template_meta = item.get("template_metadata") or {}\n            template_matches = bool(template) and (\n                operation_shape is None or template_meta.get("shape_key") == operation_shape\n            )\n            details["schema_template_available"] = bool(template)\n            details["shape_match"] = bool(exact_exemplar or template_matches)\n\n            # V3.1.1 precedence: a value-free template outranks any concrete prior output.\n            if template_matches:\n                lines.append(\n                    "  MEMORY TYPE: SCHEMA_TEMPLATE (persistent, deterministic, value-free, not scorer-verified; "\n                    "replace every symbolic placeholder with exact values from the CURRENT issue brief)\\n"\n                    + template.strip()\n                )\n                details["schema_template_retrieved"] = True\n                details["persistent_schema_template_retrieved"] = True\n                details["template_hash_at_decision"] = item.get("schema_template_hash")\n                details["template_source_at_decision"] = template_meta.get("source")\n                details["memory_provenance"] = "SCHEMA_TEMPLATE"\n                details["primary_memory_class"] = "PERSISTENT_SCHEMA_TEMPLATE"\n                details["verified_exemplar_suppressed_by_template"] = bool(exact_exemplar)\n            elif structural:\n                lines.append(\n                    "  MEMORY TYPE: VERIFIED_STRUCTURAL_EXEMPLAR (derived from a scorer-passing implementation, "\n                    "but value-free; replace placeholders with CURRENT issue values)\\n"\n                    + structural.strip()\n                )\n                details["verified_exemplar_retrieved"] = True\n                details["verified_structural_exemplar_retrieved"] = True\n                details["memory_provenance"] = "VERIFIED_STRUCTURAL_EXEMPLAR"\n                details["primary_memory_class"] = "VERIFIED_STRUCTURAL_EXEMPLAR"\n            elif exact_exemplar:\n                lines.append(\n                    "  MEMORY TYPE: RAW_VERIFIED_EXEMPLAR (scorer-passing historical output; use only its structure "\n                    "and replace all values from the CURRENT issue brief)\\n"\n                    + exact_exemplar["content"].strip()\n                )\n                details["verified_exemplar_retrieved"] = True\n                details["raw_verified_exemplar_retrieved"] = True\n                details["memory_provenance"] = "RAW_VERIFIED_EXEMPLAR"\n                details["primary_memory_class"] = "RAW_VERIFIED_EXEMPLAR"\n        lines.append("=== END AKILI PROFILE ===")\n        block, used = truncate_to_budget("\\n".join(lines), self.budget, self.count)\n        retrieved_key = profile["key"]\n        if retrieved_key != self.current_key:\n            self.cross_scope_retrievals += 1\n        row = {\n            "episode": self.episode_count, "requested": self.current_key,\n            "retrieved": retrieved_key, "tokens": used, **details,\n        }\n        self.retrieval_log.append(row)\n        self.retrieval_log = self.retrieval_log[-self.max_retrieval_log:]\n        self._audit("PROFILE_RETRIEVED", {"retrieved": retrieved_key, "tokens": used, **details})\n        return block, details\n\n    def memory_prompt(self):\n        return self.retrieval_context()[0]\n\n    def snapshot(self):\n        return {\n            "instance_id": self.instance_id, "current_key": self.current_key,\n            "profiles": copy.deepcopy(self.profiles), "retrieval_log": copy.deepcopy(self.retrieval_log),\n        }\n\n    def report(self):\n        live = {"PROVISIONAL", "VERIFIED", "CONFLICTED"}\n        admitted_total = sum(1 for p in self.profiles.values() for v in p["versions"]\n                             if v["status"] in live | {"SUPERSEDED"})\n        superseded = sum(1 for p in self.profiles.values() for v in p["versions"] if v["status"] == "SUPERSEDED")\n        active_rules = sum(len(p["active_by_family"]) for p in self.profiles.values())\n        lifecycle_counts = {state: 0 for state in ["PROVISIONAL", "VERIFIED", "CONFLICTED", "REJECTED"]}\n        status_counts = {state: 0 for state in ["PROVISIONAL", "VERIFIED", "CONFLICTED", "SUPERSEDED", "REJECTED"]}\n        schema_templates = 0\n        rejected_schema_templates = 0\n        verified_exemplars = 0\n        for p in self.profiles.values():\n            for v in p["versions"]:\n                lifecycle = v.get("lifecycle_state", v["status"])\n                lifecycle_counts[lifecycle] = lifecycle_counts.get(lifecycle, 0) + 1\n                status_counts[v["status"]] = status_counts.get(v["status"], 0) + 1\n                if v.get("status") == "REJECTED":\n                    rejected_schema_templates += int(bool(v.get("schema_template")))\n                else:\n                    schema_templates += int(bool(v.get("schema_template")))\n                    verified_exemplars += len(v.get("verified_exemplars") or {})\n        state_blob = json.dumps(self.snapshot(), sort_keys=True, default=str)\n        state_bounded = (\n            len(self.profiles) <= self.max_profiles and len(self.retrieval_log) <= self.max_retrieval_log\n            and all(len(p["versions"]) <= self.max_versions for p in self.profiles.values())\n            and all(len(p["stats"]) <= self.max_stats_keys for p in self.profiles.values())\n        )\n        return {\n            "instance_id": self.instance_id, "profiles": len(self.profiles),\n            "stored_memory_chars": len(state_blob), "stored_memory_tokens": self.count(state_blob),\n            "active_rules": active_rules, "admitted_total": admitted_total, "superseded": superseded,\n            "lifecycle_counts": lifecycle_counts, "status_counts": status_counts,\n            "schema_templates": schema_templates,\n            "rejected_schema_templates": rejected_schema_templates,\n            "verified_exemplars": verified_exemplars,\n            "audit_events": len(self.audit), "audit_valid": self.validate_chain(),\n            "cross_scope_retrievals": self.cross_scope_retrievals, "state_bounded": state_bounded,\n            "current_entity": self.current_key,\n        }\n'
core_path = WORK_ROOT / "akili_v311a_core.py"
core_path.write_text(CORE_SOURCE, encoding="utf-8")
print("core sha256:", hashlib.sha256(core_path.read_bytes()).hexdigest())


In [ ]:
# Embedded benchmark source.
BENCHMARK_SOURCE = '#!/usr/bin/env python3\n"""Akili V4 external structural-generalisation benchmark.\n\nFour arms: stateless, bounded ICL, immediate-template-only, and full Akili.\nSix operation families are structurally distinct from the V3.1.1a clamp/wrap\nfamilies. Development and held-out seeds are phase locked.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport copy\nimport dataclasses\nimport hashlib\nimport json\nimport os\nimport random\nimport re\nimport statistics\nimport tempfile\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any, Callable, Dict, Iterable, List, Optional, Tuple\n\nfrom akili_v311a_core import AkiliCore, BoundedICL, Stateless, truncate_to_budget\n\nPROTOCOL = "akili-v4-external-structural-generalisation-v1"\nLOCKED_PHASE_SEEDS = {"development": [1, 2, 3], "heldout": [4, 5, 6]}\nARMS = ["stateless", "bounded_icl", "template_only", "akili"]\nFAMILIES = [\n    "validation_exception",\n    "serialization_mapping",\n    "structured_logging",\n    "config_precedence",\n    "batch_tail_policy",\n    "artifact_naming",\n]\n\n\ndef canonical_hash(obj: Any) -> str:\n    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()\n\n\ndef token_count_approx(text: str) -> int:\n    return max(0, (len(text or "") + 3) // 4)\n\n\ndef operation_shape(family: str) -> str:\n    return {\n        "validation_exception": "unary_range_validation",\n        "serialization_mapping": "record_to_dict_mapping",\n        "structured_logging": "event_to_structured_log",\n        "config_precedence": "first_non_null_precedence",\n        "batch_tail_policy": "list_chunking_tail_policy",\n        "artifact_naming": "path_string_naming",\n    }[family]\n\n\ndef render_template(family: str) -> str:\n    templates = {\n        "validation_exception": \'\'\'def validate_value(value):\n    if value < <LOWER_BOUND> or value > <UPPER_BOUND>:\n        raise <EXCEPTION_TYPE>(<MESSAGE>)\n    return value\'\'\',\n        "serialization_mapping": \'\'\'def serialize_record(record):\n    return {\n        <OUTPUT_KEY_1>: record[<INPUT_KEY_1>],\n        <OUTPUT_KEY_2>: record[<INPUT_KEY_2>],\n    }\'\'\',\n        "structured_logging": \'\'\'def build_log(event, tenant):\n    return {\n        "level": <LEVEL>,\n        "message": <PREFIX> + event,\n        "context": {"tenant": tenant, "source": <SOURCE>},\n    }\'\'\',\n        "config_precedence": \'\'\'def resolve_config(cli, env, file_value, default):\n    for value in (<SOURCE_1>, <SOURCE_2>, <SOURCE_3>, <SOURCE_4>):\n        if value is not None:\n            return value\n    return None\'\'\',\n        "batch_tail_policy": \'\'\'def make_batches(items):\n    batches = [items[index:index + <BATCH_SIZE>] for index in range(0, len(items), <BATCH_SIZE>)]\n    if <DROP_INCOMPLETE> and batches and len(batches[-1]) < <BATCH_SIZE>:\n        batches.pop()\n    return batches\'\'\',\n        "artifact_naming": \'\'\'def artifact_path(root, project, name):\n    return f"{root}/{project}/<PREFIX>_{name}.<EXTENSION>"\'\'\',\n    }\n    return templates[family]\n\n\ndef current_template_memory(family: str) -> str:\n    return (\n        "MEMORY TYPE: CURRENT_INSTRUCTION_TEMPLATE\\n"\n        "Source: deterministic renderer from the current maintainer-confirmed convention.\\n"\n        "Status: structural and value-free; replace placeholders using CURRENT BRIEF JSON only.\\n"\n        + render_template(family)\n    )\n\n\ndef public_rule(issue: Dict[str, Any]) -> str:\n    family, p = issue["family"], issue["params"]\n    if family == "validation_exception":\n        return f"validate inclusively within [{p[\'low\']}, {p[\'high\']}], otherwise raise {p[\'exception\']} with message {p[\'message\']!r}"\n    if family == "serialization_mapping":\n        return f"map input keys {p[\'in1\']!r},{p[\'in2\']!r} to output keys {p[\'out1\']!r},{p[\'out2\']!r}"\n    if family == "structured_logging":\n        return f"return level {p[\'level\']!r}, prefix {p[\'prefix\']!r}, source {p[\'source\']!r}, and tenant context"\n    if family == "config_precedence":\n        return "choose the first non-None source in order " + " > ".join(p["order"])\n    if family == "batch_tail_policy":\n        return f"batch size {p[\'size\']} and drop_incomplete={p[\'drop\']}"\n    if family == "artifact_naming":\n        return f"path root/project/{p[\'prefix\']}_name.{p[\'ext\']}"\n    raise KeyError(family)\n\n\ndef expected_code(issue: Dict[str, Any]) -> str:\n    p = issue["params"]\n    f = issue["family"]\n    if f == "validation_exception":\n        return (\n            "def validate_value(value):\\n"\n            f"    if value < {p[\'low\']!r} or value > {p[\'high\']!r}:\\n"\n            f"        raise {p[\'exception\']}({p[\'message\']!r})\\n"\n            "    return value\\n"\n        )\n    if f == "serialization_mapping":\n        return (\n            "def serialize_record(record):\\n"\n            "    return {\\n"\n            f"        {p[\'out1\']!r}: record[{p[\'in1\']!r}],\\n"\n            f"        {p[\'out2\']!r}: record[{p[\'in2\']!r}],\\n"\n            "    }\\n"\n        )\n    if f == "structured_logging":\n        return (\n            "def build_log(event, tenant):\\n"\n            "    return {\\n"\n            f"        \'level\': {p[\'level\']!r},\\n"\n            f"        \'message\': {p[\'prefix\']!r} + event,\\n"\n            f"        \'context\': {{\'tenant\': tenant, \'source\': {p[\'source\']!r}}},\\n"\n            "    }\\n"\n        )\n    if f == "config_precedence":\n        return (\n            "def resolve_config(cli, env, file_value, default):\\n"\n            f"    for value in ({\', \'.join(p[\'order\'])}):\\n"\n            "        if value is not None:\\n"\n            "            return value\\n"\n            "    return None\\n"\n        )\n    if f == "batch_tail_policy":\n        return (\n            "def make_batches(items):\\n"\n            f"    batches = [items[index:index + {p[\'size\']}] for index in range(0, len(items), {p[\'size\']})]\\n"\n            f"    if {p[\'drop\']} and batches and len(batches[-1]) < {p[\'size\']}:\\n"\n            "        batches.pop()\\n"\n            "    return batches\\n"\n        )\n    if f == "artifact_naming":\n        return (\n            "def artifact_path(root, project, name):\\n"\n            f"    return f\\"{{root}}/{{project}}/{p[\'prefix\']}_{{name}}.{p[\'ext\']}\\"\\n"\n        )\n    raise KeyError(f)\n\n\ndef family_params(family: str, version: int, seed: int) -> Dict[str, Any]:\n    rng = random.Random(seed * 10_000 + FAMILIES.index(family) * 100 + version)\n    if family == "validation_exception":\n        low = rng.choice([-10, -5, 0, 1]) + version\n        high = low + rng.choice([7, 10, 15])\n        return {"low": low, "high": high, "exception": "ValueError" if version % 2 else "RuntimeError", "message": f"range-v{version}"}\n    if family == "serialization_mapping":\n        return {"in1": f"id_v{version}", "in2": f"kind_v{version}", "out1": f"identifier_v{version}", "out2": f"category_v{version}"}\n    if family == "structured_logging":\n        return {"level": ["INFO", "WARNING", "ERROR"][version % 3], "prefix": f"v{version}:", "source": f"service-{rng.randrange(10,99)}"}\n    if family == "config_precedence":\n        orders = [\n            ["cli", "env", "file_value", "default"],\n            ["env", "cli", "file_value", "default"],\n            ["file_value", "env", "cli", "default"],\n        ]\n        return {"order": orders[(version - 1) % len(orders)]}\n    if family == "batch_tail_policy":\n        return {"size": 2 + ((version + seed) % 4), "drop": bool(version % 2)}\n    if family == "artifact_naming":\n        return {"prefix": ["build", "artifact", "release"][version % 3], "ext": ["json", "txt", "dat"][seed % 3]}\n    raise KeyError(family)\n\n\ndef make_env(seed: int) -> Dict[str, Any]:\n    scopes = ["atlas", "beacon", "cedar"]\n    family_scope = {family: scopes[i // 2] for i, family in enumerate(FAMILIES)}\n    stages: List[List[Tuple[str, int, bool, str]]] = []\n    # Two teach issues per version meet the frozen evidence threshold of two.\n    stages.append([(f, 1, True, "acquisition_1") for f in FAMILIES])\n    stages.append([(f, 1, True, "acquisition_2") for f in reversed(FAMILIES)])\n    stages.append([(f, 1, False, "return_after_interruption") for f in FAMILIES])\n    stages.append([(f, 2, True, "supersession_1") for f in reversed(FAMILIES)])\n    stages.append([(f, 2, True, "supersession_2") for f in FAMILIES])\n    stages.append([(f, 2, False, "post_supersession_return") for f in reversed(FAMILIES)])\n    rng = random.Random(seed)\n    issues: List[Dict[str, Any]] = []\n    issue_id = 0\n    for stage_index, stage in enumerate(stages):\n        stage = list(stage)\n        rng.shuffle(stage)\n        for family, version, teach, slice_name in stage:\n            issue_id += 1\n            params = family_params(family, version, seed)\n            issue = {\n                "id": issue_id,\n                "seed": seed,\n                "scope": family_scope[family],\n                "family": family,\n                "shape": operation_shape(family),\n                "version": version,\n                "teach": teach,\n                "slice": slice_name,\n                "params": params,\n                "rule": public_rule({"family": family, "params": params}),\n            }\n            issue["expected_code"] = expected_code(issue)\n            issue["brief_json"] = json.dumps({\n                "scope": issue["scope"], "family": family, "version": version,\n                "parameters": params,\n            }, sort_keys=True)\n            issues.append(issue)\n    env = {"seed": seed, "issues": issues, "families": FAMILIES, "scopes": scopes}\n    env["environment_hash"] = canonical_hash(env)\n    return env\n\n\nALLOWED_CALLS = {"range", "len"}\nFORBIDDEN_NODES = (ast.Import, ast.ImportFrom, ast.While, ast.With, ast.AsyncWith, ast.Lambda, ast.ClassDef, ast.Global, ast.Nonlocal, ast.Delete, ast.Yield, ast.YieldFrom, ast.Await)\n\n\ndef normalize_content(content: str) -> str:\n    text = (content or "").strip()\n    if text.startswith("```"):\n        lines = text.splitlines()\n        if lines and lines[0].startswith("```"):\n            lines = lines[1:]\n        if lines and lines[-1].strip() == "```":\n            lines = lines[:-1]\n        text = "\\n".join(lines).strip()\n    return text\n\n\ndef safe_ast(code: str, expected_name: str) -> Tuple[bool, str]:\n    try:\n        tree = ast.parse(code)\n    except SyntaxError as exc:\n        return False, f"syntax:{exc}"\n    if len(tree.body) != 1 or not isinstance(tree.body[0], ast.FunctionDef):\n        return False, "must contain exactly one function"\n    fn = tree.body[0]\n    if fn.name != expected_name:\n        return False, f"wrong function name {fn.name}"\n    for node in ast.walk(tree):\n        if isinstance(node, FORBIDDEN_NODES):\n            return False, f"forbidden node {type(node).__name__}"\n        if isinstance(node, ast.Attribute) and node.attr.startswith("__"):\n            return False, "dunder attribute forbidden"\n        if isinstance(node, ast.Name) and node.id.startswith("__"):\n            return False, "dunder name forbidden"\n        if isinstance(node, ast.Call):\n            if isinstance(node.func, ast.Name):\n                if node.func.id not in ALLOWED_CALLS and node.func.id not in {"ValueError", "RuntimeError"}:\n                    return False, f"call {node.func.id} forbidden"\n            elif isinstance(node.func, ast.Attribute):\n                if node.func.attr not in {"pop"}:\n                    return False, f"method {node.func.attr} forbidden"\n            else:\n                return False, "dynamic call forbidden"\n    return True, "ok"\n\n\ndef score_code(issue: Dict[str, Any], content: str) -> Tuple[bool, str]:\n    code = normalize_content(content)\n    expected_name = {\n        "validation_exception": "validate_value",\n        "serialization_mapping": "serialize_record",\n        "structured_logging": "build_log",\n        "config_precedence": "resolve_config",\n        "batch_tail_policy": "make_batches",\n        "artifact_naming": "artifact_path",\n    }[issue["family"]]\n    ok, reason = safe_ast(code, expected_name)\n    if not ok:\n        return False, reason\n    namespace: Dict[str, Any] = {"__builtins__": {"range": range, "len": len, "ValueError": ValueError, "RuntimeError": RuntimeError}}\n    try:\n        exec(compile(code, "<candidate>", "exec"), namespace, namespace)\n        fn = namespace[expected_name]\n        p = issue["params"]\n        if issue["family"] == "validation_exception":\n            midpoint = (p["low"] + p["high"]) / 2\n            if fn(midpoint) != midpoint:\n                return False, "in-range result"\n            exc_cls = ValueError if p["exception"] == "ValueError" else RuntimeError\n            for value in (p["low"] - 1, p["high"] + 1):\n                try:\n                    fn(value)\n                    return False, "missing exception"\n                except exc_cls as exc:\n                    if str(exc) != p["message"]:\n                        return False, "wrong message"\n        elif issue["family"] == "serialization_mapping":\n            record = {p["in1"]: 7, p["in2"]: "x", "noise": 9}\n            if fn(record) != {p["out1"]: 7, p["out2"]: "x"}:\n                return False, "mapping mismatch"\n        elif issue["family"] == "structured_logging":\n            if fn("started", "tenant-a") != {"level": p["level"], "message": p["prefix"] + "started", "context": {"tenant": "tenant-a", "source": p["source"]}}:\n                return False, "log mismatch"\n        elif issue["family"] == "config_precedence":\n            values = {"cli": "CLI", "env": "ENV", "file_value": "FILE", "default": "DEF"}\n            args = [values[x] for x in ["cli", "env", "file_value", "default"]]\n            if fn(*args) != values[p["order"][0]]:\n                return False, "precedence mismatch"\n            values[p["order"][0]] = None\n            args = [values[x] for x in ["cli", "env", "file_value", "default"]]\n            if fn(*args) != values[p["order"][1]]:\n                return False, "fallback precedence mismatch"\n        elif issue["family"] == "batch_tail_policy":\n            items = list(range(p["size"] * 2 + 1))\n            expected = [items[:p["size"]], items[p["size"]:p["size"]*2]]\n            if not p["drop"]:\n                expected.append(items[p["size"]*2:])\n            if fn(items) != expected:\n                return False, "batch mismatch"\n        elif issue["family"] == "artifact_naming":\n            if fn("/tmp", "alpha", "model") != f"/tmp/alpha/{p[\'prefix\']}_model.{p[\'ext\']}":\n                return False, "path mismatch"\n    except Exception as exc:\n        return False, f"runtime:{type(exc).__name__}:{exc}"\n    return True, "pass"\n\n\nclass TemplateOnly:\n    name = "template_only"\n    instance_id = "template-only"\n    def switch_entity(self, _name: str) -> None: pass\n    def observe_episode(self, *_args: Any, **_kwargs: Any) -> None: pass\n    def report(self) -> Dict[str, Any]:\n        return {"profiles": 0, "stored_memory_tokens": 0, "cross_scope_retrievals": 0, "state_bounded": True, "audit_valid": True}\n\n\ndef derive_v4(_stats: Dict[str, Any], _profile: Dict[str, Any], features: Dict[str, Any]):\n    if not features.get("teach"):\n        return None\n    return {\n        "family": features["family"],\n        "rule": features["rule"],\n        "decision_label": f"v{features[\'version\']}",\n        "lifecycle_state": "VERIFIED",\n        "evidence": {"count": features["evidence_count"], "version": features["version"]},\n        "admit_threshold": 2,\n        "schema_template": render_template(features["family"]),\n        "template_metadata": {\n            "source": "v4_deterministic_public_metadata_renderer",\n            "shape_key": operation_shape(features["family"]),\n            "value_free": True,\n        },\n    }\n\n\ndef prompt_for(issue: Dict[str, Any], memory: str) -> Tuple[str, str]:\n    system = (\n        "You are a repository implementation agent. Return one JSON object with key content. "\n        "content must contain exactly one Python function and no tests or prose."\n    )\n    teach = ""\n    if issue["teach"]:\n        teach = "\\nCURRENT MAINTAINER CONVENTION:\\n" + issue["rule"] + "\\n"\n    user = (\n        f"WORKSPACE: {issue[\'scope\']}\\nFAMILY: {issue[\'family\']}\\n"\n        + teach\n        + "CURRENT BRIEF JSON:\\n" + issue["brief_json"] + "\\n"\n        + ("ADAPTATION MEMORY:\\n" + memory + "\\n" if memory else "")\n        + "Implement the required function now."\n    )\n    return system, user\n\n\nclass MockModel:\n    """Pipeline oracle. It intentionally needs a structural template."""\n    def __init__(self):\n        self.calls = 0\n    def __call__(self, _system: str, user: str, _schema: str, issue: Dict[str, Any]) -> Dict[str, Any]:\n        self.calls += 1\n        has_template = "CURRENT_INSTRUCTION_TEMPLATE" in user or "SCHEMA_TEMPLATE" in user\n        if has_template:\n            return {"content": issue["expected_code"], "prompt_tokens": token_count_approx(user), "generated_tokens": token_count_approx(issue["expected_code"]), "strict_json": True, "usable": True, "attempts": 1}\n        return {"content": "def placeholder(*args):\\n    return None\\n", "prompt_tokens": token_count_approx(user), "generated_tokens": 8, "strict_json": True, "usable": True, "attempts": 1}\n\n\nclass HFModel:\n    def __init__(self, model_name: str):\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n        self.torch = torch\n        assert torch.cuda.is_available(), "CUDA GPU required for real mode"\n        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)\n        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id\n        self.model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", quantization_config=quant, torch_dtype=dtype)\n        self.model.eval()\n    def count(self, text: str) -> int:\n        return len(self.tokenizer.encode(text or "", add_special_tokens=False))\n    def _parse(self, raw: str) -> Tuple[Optional[Dict[str, Any]], bool]:\n        text = raw.strip()\n        try:\n            obj = json.loads(text)\n            return (obj if isinstance(obj, dict) and isinstance(obj.get("content"), str) else None), True\n        except Exception:\n            pass\n        decoder = json.JSONDecoder()\n        for match in re.finditer(r"\\{", text):\n            try:\n                obj, _ = decoder.raw_decode(text[match.start():])\n                if isinstance(obj, dict) and isinstance(obj.get("content"), str):\n                    return obj, False\n            except Exception:\n                continue\n        # Safe recovery for a plain Python function.\n        lines = text.splitlines()\n        for end in range(len(lines), 0, -1):\n            candidate = "\\n".join(lines[:end]).strip()\n            try:\n                tree = ast.parse(candidate)\n                if len(tree.body) == 1 and isinstance(tree.body[0], ast.FunctionDef):\n                    return {"content": candidate}, False\n            except Exception:\n                pass\n        return None, False\n    def __call__(self, system: str, user: str, _schema: str, issue: Dict[str, Any]) -> Dict[str, Any]:\n        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]\n        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)\n        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)\n        attempts = 0\n        total_prompt = total_generated = 0\n        raw_outputs = []\n        for retry in range(2):\n            attempts += 1\n            total_prompt += int(inputs["input_ids"].shape[-1])\n            with self.torch.inference_mode():\n                output = self.model.generate(**inputs, max_new_tokens=420, do_sample=False, pad_token_id=self.tokenizer.pad_token_id)\n            generated = output[0, inputs["input_ids"].shape[-1]:]\n            total_generated += int(generated.shape[-1])\n            raw = self.tokenizer.decode(generated, skip_special_tokens=True)\n            raw_outputs.append(raw)\n            obj, strict = self._parse(raw)\n            if obj is not None:\n                return {"content": obj["content"], "prompt_tokens": total_prompt, "generated_tokens": total_generated, "strict_json": strict, "usable": True, "attempts": attempts, "raw_outputs": raw_outputs}\n            retry_user = user + "\\nYour prior answer was invalid. Return only one JSON object: {\\"content\\": \\"def ...\\"}."\n            messages[-1]["content"] = retry_user\n            prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)\n            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)\n        return {"content": "", "prompt_tokens": total_prompt, "generated_tokens": total_generated, "strict_json": False, "usable": False, "attempts": attempts, "raw_outputs": raw_outputs}\n\n\ndef run_arm(env: Dict[str, Any], arm: str, model: Any, budget_icl: int, budget_akili: int) -> Dict[str, Any]:\n    count_fn = getattr(model, "count", token_count_approx)\n    if arm == "stateless": system_obj: Any = Stateless(count_fn)\n    elif arm == "bounded_icl": system_obj = BoundedICL(budget_icl, count_fn)\n    elif arm == "template_only": system_obj = TemplateOnly()\n    elif arm == "akili":\n        system_obj = AkiliCore(budget_akili, count_fn, derive_v4, min_evidence=2, max_profiles=8, max_versions=64, recent_window=10, max_stats_keys=96, max_retrieval_log=512)\n    else: raise KeyError(arm)\n\n    evidence_counts: Dict[Tuple[str, str, int], int] = defaultdict(int)\n    traces = []\n    old_versions: Dict[Tuple[str, str], int] = {}\n    for issue in env["issues"]:\n        if hasattr(system_obj, "switch_entity"):\n            system_obj.switch_entity(issue["scope"])\n        memory = ""\n        provenance = "NO_MEMORY"\n        details: Dict[str, Any] = {}\n        if arm == "bounded_icl":\n            memory = system_obj.memory_prompt()\n            provenance = "RAW_HISTORY"\n        elif arm == "template_only" and issue["teach"]:\n            memory = current_template_memory(issue["family"])\n            provenance = "CURRENT_INSTRUCTION_TEMPLATE"\n        elif arm == "akili":\n            if issue["teach"]:\n                memory = current_template_memory(issue["family"])\n                provenance = "CURRENT_INSTRUCTION_TEMPLATE"\n                details = {"current_instruction_template_retrieved": True}\n            else:\n                memory, details = system_obj.retrieval_context(issue["family"], issue["shape"])\n                provenance = details.get("primary_memory_class", "NO_MEMORY")\n        memory, memory_tokens = truncate_to_budget(memory, budget_akili if arm in {"akili", "template_only"} else budget_icl, count_fn)\n        sys_prompt, user_prompt = prompt_for(issue, memory)\n        response = model(sys_prompt, user_prompt, "content", issue)\n        passed, reason = score_code(issue, response.get("content", ""))\n        if arm == "bounded_icl":\n            system_obj.observe_episode(\n                summary=(\n                    user_prompt + "\\nASSISTANT OUTPUT:\\n" + response.get("content", "")\n                    + f"\\nVERIFIER: {\'PASS\' if passed else \'FAIL\'}"\n                ),\n                features={},\n                entity_hint=issue["scope"], outcome="PASS" if passed else "FAIL",\n            )\n        elif arm == "akili":\n            key = (issue["scope"], issue["family"], issue["version"])\n            if issue["teach"]:\n                evidence_counts[key] += 1\n            system_obj.observe_episode(\n                summary=f"{issue[\'family\']} v{issue[\'version\']}",\n                features={\n                    "teach": issue["teach"], "family": issue["family"], "version": issue["version"],\n                    "rule": issue["rule"], "evidence_count": evidence_counts[key],\n                    f"latest_{issue[\'family\']}_version": issue["version"],\n                },\n                entity_hint=issue["scope"], outcome="PASS" if passed else "FAIL",\n            )\n        prior = old_versions.get((issue["scope"], issue["family"]))\n        if issue["teach"] and evidence_counts.get((issue["scope"], issue["family"], issue["version"]), 0) >= 2:\n            old_versions[(issue["scope"], issue["family"])] = issue["version"]\n        trace = {\n            "issue_id": issue["id"], "seed": issue["seed"], "arm": arm, "scope": issue["scope"],\n            "family": issue["family"], "version": issue["version"], "teach": issue["teach"],\n            "slice": issue["slice"], "passed": passed, "reason": reason,\n            "memory_provenance": provenance, "memory_tokens": memory_tokens,\n            "prompt_tokens": response.get("prompt_tokens", 0), "generated_tokens": response.get("generated_tokens", 0),\n            "usable": response.get("usable", False), "strict_json": response.get("strict_json", False),\n            "attempts": response.get("attempts", 0), "cross_scope_retrievals": getattr(system_obj, "cross_scope_retrievals", 0),\n            "details": details,\n        }\n        active_label = None\n        if arm == "akili" and not issue["teach"]:\n            active = system_obj.active_record(issue["family"])\n            active_label = (active or {}).get("decision_label")\n        trace["active_decision_label"] = active_label\n        trace["obsolete_memory"] = bool(\n            arm == "akili" and not issue["teach"] and active_label != f"v{issue[\'version\']}"\n        )\n        traces.append(trace)\n    report = system_obj.report()\n    return {"arm": arm, "traces": traces, "system_report": report}\n\n\ndef aggregate(run_data: Dict[int, Dict[str, Any]]) -> Dict[str, Any]:\n    result: Dict[str, Any] = {"arms": {}, "integrity": {}, "performance": {}}\n    for arm in ARMS:\n        traces = [t for seed in run_data.values() for t in seed[arm]["traces"]]\n        passed = sum(t["passed"] for t in traces)\n        by_slice: Dict[str, Dict[str, Any]] = {}\n        for slice_name in sorted({t["slice"] for t in traces}):\n            rows = [t for t in traces if t["slice"] == slice_name]\n            by_slice[slice_name] = {"issues": len(rows), "passed": sum(r["passed"] for r in rows), "rate": sum(r["passed"] for r in rows) / len(rows)}\n        result["arms"][arm] = {\n            "issues": len(traces), "passed": passed, "accuracy": passed / len(traces),\n            "prompt_tokens": sum(t["prompt_tokens"] for t in traces),\n            "generated_tokens": sum(t["generated_tokens"] for t in traces),\n            "memory_tokens": sum(t["memory_tokens"] for t in traces),\n            "retries": sum(max(0, t["attempts"] - 1) for t in traces),\n            "usable_rate": sum(t["usable"] for t in traces) / len(traces),\n            "strict_first_rate": sum(t["strict_json"] for t in traces) / len(traces),\n            "obsolete_retrievals": sum(t["obsolete_memory"] for t in traces),\n            "by_slice": by_slice,\n        }\n    all_traces = [t for seed in run_data.values() for arm in ARMS for t in seed[arm]["traces"]]\n    integrity = {\n        "three_seed_locked_design": len(run_data) == 3,\n        "all_arm_issue_counts_equal": len({len(run_data[s][a]["traces"]) for s in run_data for a in ARMS}) == 1,\n        "environment_hashes_present": all(seed_data.get("environment_hash") for seed_data in run_data.values()),\n        "cross_scope_retrieval_zero": all(t["cross_scope_retrievals"] == 0 for t in all_traces),\n        "obsolete_retrieval_zero": all(not t["obsolete_memory"] for t in all_traces if t["arm"] == "akili"),\n        "akili_teaching_template_coverage": all(t["memory_provenance"] == "CURRENT_INSTRUCTION_TEMPLATE" for t in all_traces if t["arm"] == "akili" and t["teach"]),\n        "template_only_has_no_persistent_memory": all(t["memory_provenance"] in {"CURRENT_INSTRUCTION_TEMPLATE", "NO_MEMORY"} for t in all_traces if t["arm"] == "template_only"),\n        "akili_recall_uses_persistent_template": all(t["memory_provenance"] == "PERSISTENT_SCHEMA_TEMPLATE" for t in all_traces if t["arm"] == "akili" and not t["teach"]),\n        "all_system_audits_valid": all(run_data[s][a]["system_report"].get("audit_valid", True) for s in run_data for a in ARMS),\n        "all_system_states_bounded": all(run_data[s][a]["system_report"].get("state_bounded", True) for s in run_data for a in ARMS),\n    }\n    result["integrity"] = integrity\n    ak = result["arms"]["akili"]\n    icl = result["arms"]["bounded_icl"]\n    prompt_reduction = 1 - ak["prompt_tokens"] / max(icl["prompt_tokens"], 1)\n    perf = {\n        "akili_accuracy_at_least_90": ak["accuracy"] >= 0.90,\n        "akili_within_5pp_of_icl": ak["accuracy"] + 0.05 >= icl["accuracy"],\n        "akili_acquisition_at_least_90": statistics.fmean([ak["by_slice"][x]["rate"] for x in ("acquisition_1", "acquisition_2")]) >= 0.90,\n        "akili_return_at_least_90": ak["by_slice"]["return_after_interruption"]["rate"] >= 0.90,\n        "akili_post_supersession_at_least_90": ak["by_slice"]["post_supersession_return"]["rate"] >= 0.90,\n        "prompt_reduction_at_least_70": prompt_reduction >= 0.70,\n        "obsolete_retrieval_zero": ak["obsolete_retrievals"] == 0,\n        "cross_scope_retrieval_zero": integrity["cross_scope_retrieval_zero"],\n        "all_integrity_pass": all(integrity.values()),\n    }\n    result["performance"] = {"criteria": perf, "prompt_reduction": prompt_reduction, "PASS": all(perf.values())}\n    return result\n\n\ndef run(phase: str, mode: str, output: Path, model_name: str, budget_icl: int, budget_akili: int) -> Dict[str, Any]:\n    assert phase in LOCKED_PHASE_SEEDS\n    seeds = LOCKED_PHASE_SEEDS[phase]\n    output.mkdir(parents=True, exist_ok=True)\n    model: Any = MockModel() if mode == "mock" else HFModel(model_name)\n    run_data: Dict[int, Dict[str, Any]] = {}\n    for seed in seeds:\n        env = make_env(seed)\n        run_data[seed] = {"environment_hash": env["environment_hash"]}\n        for arm in ARMS:\n            print(f"seed={seed} arm={arm}", flush=True)\n            run_data[seed][arm] = run_arm(env, arm, model, budget_icl, budget_akili)\n            (output / f"seed{seed}_{arm}.json").write_text(json.dumps(run_data[seed][arm], indent=2), encoding="utf-8")\n    summary = aggregate(run_data)\n    receipt = {\n        "protocol": PROTOCOL, "phase": phase, "mode": mode, "seeds": seeds,\n        "model": model_name if mode == "real" else "prompt-path mock",\n        "budgets": {"bounded_icl": budget_icl, "akili": budget_akili},\n        "environment_hashes": {str(s): run_data[s]["environment_hash"] for s in seeds},\n        "summary": summary,\n    }\n    receipt["implementation_fingerprint"] = canonical_hash({\n        "protocol": PROTOCOL,\n        "families": FAMILIES,\n        "templates": {f: render_template(f) for f in FAMILIES},\n        "scorer_source": ast.get_source_segment(open(__file__, encoding="utf-8").read(), ast.parse(open(__file__, encoding="utf-8").read()).body[0]) if False else "see file hash",\n    })\n    receipt["script_sha256"] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    (output / "FINAL_RECEIPT.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")\n    print(json.dumps(summary, indent=2))\n    return receipt\n\n\ndef parse_args(argv: Optional[Iterable[str]] = None) -> argparse.Namespace:\n    p = argparse.ArgumentParser()\n    p.add_argument("--phase", choices=LOCKED_PHASE_SEEDS, default=os.getenv("AKILI_V4_PHASE", "development"))\n    p.add_argument("--mode", choices=["mock", "real"], default=os.getenv("AKILI_V4_MODE", "mock"))\n    p.add_argument("--output", type=Path, default=Path(os.getenv("AKILI_V4_OUTPUT", "./akili_v4_output")))\n    p.add_argument("--model", default=os.getenv("AKILI_V4_MODEL", "Qwen/Qwen3-4B"))\n    p.add_argument("--icl-budget", type=int, default=int(os.getenv("AKILI_V4_ICL_BUDGET", "1024")))\n    p.add_argument("--akili-budget", type=int, default=int(os.getenv("AKILI_V4_AKILI_BUDGET", "288")))\n    return p.parse_args(argv)\n\n\nif __name__ == "__main__":\n    args = parse_args()\n    receipt = run(args.phase, args.mode, args.output, args.model, args.icl_budget, args.akili_budget)\n    raise SystemExit(0 if receipt["summary"]["integrity"] and all(receipt["summary"]["integrity"].values()) else 2)\n'
script_path = WORK_ROOT / "v4_external_generalisation.py"
script_path.write_text(BENCHMARK_SOURCE, encoding="utf-8")
subprocess.run([sys.executable, "-m", "py_compile", str(core_path), str(script_path)], check=True)
print("script sha256:", hashlib.sha256(script_path.read_bytes()).hexdigest())


In [ ]:
# Mandatory offline/prompt-path preflight. This consumes no Qwen inference.
mock_out = WORK_ROOT / "mock_preflight"
subprocess.run([
    sys.executable, str(script_path), "--phase", "development", "--mode", "mock",
    "--output", str(mock_out)
], check=True, cwd=WORK_ROOT, timeout=300)
mock_receipt = json.loads((mock_out / "FINAL_RECEIPT.json").read_text())
assert all(mock_receipt["summary"]["integrity"].values()), mock_receipt["summary"]["integrity"]
print("MOCK PREFLIGHT PASS")


In [ ]:
# REAL QWEN RUN — phase locked. Do not edit seeds, prompts, templates or scores after starting.
assert PHASE == "heldout" and SEEDS == [4, 5, 6], (PHASE, SEEDS)
cmd = [
    sys.executable, str(script_path), "--phase", PHASE, "--mode", "real",
    "--output", str(OUT), "--model", MODEL
]
print("RUNNING:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True, cwd=WORK_ROOT)
receipt = json.loads((OUT / "FINAL_RECEIPT.json").read_text())
print(json.dumps(receipt, indent=2))
